<center><h1>Wang_Gao_HW3</h1></center>
<br>
<center><font size="4">DSCI 552 - Homework 3</font></center>

Name: Gao Wang  
GitHub Username: Gavinfornever  
USC ID: 8103152166

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) AReM Data

Python packages and shared settings

In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
RANDOM_STATE = 552
DATA_ROOT = Path("data") / "AReM"
SENSORS = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
STATISTICS = ["min", "max", "mean", "median", "std", "q1", "q3"]


Load and clean the AReM activity recordings

In [2]:
# Read each recording while tolerating the delimiter inconsistencies found in the supplied files.
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"AReM data folder not found: {DATA_ROOT}")

def read_recording(path):
    """Read one AReM file, ignoring metadata and tolerating its small format inconsistencies."""
    rows = []
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            values = [value.strip() for value in (line.split(",") if "," in line else line.split()) if value.strip()]
            if len(values) != 7:
                raise ValueError(f"Expected 7 values in {path}; found {len(values)} in: {line!r}")
            rows.append([float(value) for value in values])
    frame = pd.DataFrame(rows, columns=["time"] + SENSORS)
    return frame

recordings = []
for path in sorted(DATA_ROOT.glob("*/*.csv")):
    activity_folder = path.parent.name
    activity = "bending" if activity_folder.startswith("bending") else activity_folder
    dataset_number = int(re.search(r"dataset(\d+)", path.stem).group(1))
    recordings.append({
        "instance": f"{activity_folder}_{path.stem}",
        "activity": activity,
        "activity_folder": activity_folder,
        "dataset_number": dataset_number,
        "path": path,
        "values": read_recording(path),
    })

assert len(recordings) == 88, f"Expected 88 instances; found {len(recordings)}"
row_counts = pd.Series({recording["instance"]: len(recording["values"]) for recording in recordings}, name="time_points")
print(f"Loaded {len(recordings)} recordings across {len(set(r['activity'] for r in recordings))} activities.")
print("Recordings not containing exactly 480 rows in the supplied files:")
display(row_counts[row_counts != 480].to_frame())


Loaded 88 recordings across 6 activities.
Recordings not containing exactly 480 rows in the supplied files:


,time_points
sitting_dataset8,479


### (b) Required train/test split

In [3]:
# AI prompt for this code block: "Apply the homework's deterministic file-level split: datasets 1 and 2 in bending1/bending2, and datasets 1, 2, and 3 in every other activity, belong to test; all remaining recordings belong to train."
for recording in recordings:
    if recording["activity_folder"].startswith("bending"):
        is_test = recording["dataset_number"] in (1, 2)
    else:
        is_test = recording["dataset_number"] in (1, 2, 3)
    recording["split"] = "test" if is_test else "train"

split_table = pd.crosstab(
    pd.Series([r["activity"] for r in recordings], name="activity"),
    pd.Series([r["split"] for r in recordings], name="split"),
    margins=True,
)
display(split_table)
print(f"Training instances: {sum(r['split'] == 'train' for r in recordings)}; test instances: {sum(r['split'] == 'test' for r in recordings)}.")


split,test,train,All
activity,,,
bending,4,9,13
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15
All,19,69,88


Training instances: 69; test instances: 19.


### (c) Time-Domain Feature Extraction

#### i. Examples of time-domain features

Common time-domain features summarize the signal's level, spread, distribution shape, and changes over time. Examples include minimum, maximum, mean, median, variance, standard deviation, range, interquartile range, quantiles, root-mean-square amplitude, skewness, kurtosis, zero-crossing count, slope, and autocorrelation at selected lags. The required seven statistics in this assignment capture location and spread for each sensor series. A review of time-series summary statistics discusses minimum, maximum, mean, standard deviation, median, quartiles, range, IQR, skewness, and kurtosis as useful distribution summaries ([Li et al., 2020](https://bmcmedinformdecismak.biomedcentral.com/articles/10.1186/s12911-020-1063-x)).

#### ii. Extract the seven required statistics for all six sensor series

In [4]:
# AI prompt for this code block: "For each instance and each of the six sensor time series, compute minimum, maximum, mean, median, sample standard deviation, first quartile, and third quartile, then make one wide feature row per instance."
feature_rows = []
for recording in recordings:
    row = {
        "instance": recording["instance"],
        "activity": recording["activity"],
        "activity_folder": recording["activity_folder"],
        "split": recording["split"],
        "n_time_points": len(recording["values"]),
    }
    for sensor in SENSORS:
        values = recording["values"][sensor].to_numpy()
        row[f"min_{sensor}"] = np.min(values)
        row[f"max_{sensor}"] = np.max(values)
        row[f"mean_{sensor}"] = np.mean(values)
        row[f"median_{sensor}"] = np.median(values)
        row[f"std_{sensor}"] = np.std(values, ddof=1)
        row[f"q1_{sensor}"] = np.quantile(values, 0.25)
        row[f"q3_{sensor}"] = np.quantile(values, 0.75)
    feature_rows.append(row)

feature_data = pd.DataFrame(feature_rows).sort_values(["activity", "instance"]).reset_index(drop=True)
FEATURE_COLUMNS = [f"{stat}_{sensor}" for sensor in SENSORS for stat in STATISTICS]
assert feature_data.shape == (88, 5 + 42), feature_data.shape
print(f"Feature table shape: {feature_data.shape} (88 instances, 42 extracted features, and 5 identification/split columns).")
display(feature_data[["instance", "activity", "split", "n_time_points"] + FEATURE_COLUMNS[:7]].head())


Feature table shape: (88, 47) (88 instances, 42 extracted features, and 5 identification/split columns).


,instance,activity,split,n_time_points,min_avg_rss12,max_avg_rss12,mean_avg_rss12,median_avg_rss12,std_avg_rss12,q1_avg_rss12,q3_avg_rss12
0,bending1_dataset1,bending,test,480,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00
1,bending1_dataset2,bending,test,480,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67
2,bending1_dataset3,bending,train,480,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00
3,bending1_dataset4,bending,train,480,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00
4,bending1_dataset5,bending,train,480,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75


#### iii. Standard deviations and 90% bootstrap confidence intervals

In [5]:
# AI prompt for this code block: "Estimate the standard deviation of every extracted feature and use a nonparametric percentile bootstrap over the 88 independent instances to form 90% confidence intervals."
feature_matrix = feature_data[FEATURE_COLUMNS].to_numpy()
estimated_sd = feature_matrix.std(axis=0, ddof=1)

rng = np.random.default_rng(RANDOM_STATE)
n_instances = feature_matrix.shape[0]
n_bootstrap = 3000
bootstrap_indices = rng.integers(0, n_instances, size=(n_bootstrap, n_instances))
bootstrap_sd = feature_matrix[bootstrap_indices, :].std(axis=1, ddof=1)
ci_lower, ci_upper = np.percentile(bootstrap_sd, [5, 95], axis=0)

bootstrap_summary = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "estimated_standard_deviation": estimated_sd,
    "90%_bootstrap_CI_lower": ci_lower,
    "90%_bootstrap_CI_upper": ci_upper,
}).set_index("feature")
print(f"Percentile bootstrap: {n_bootstrap:,} resamples of the {n_instances} instances; interval endpoints are the 5th and 95th percentiles.")
display(bootstrap_summary.round(4))


Percentile bootstrap: 3,000 resamples of the 88 instances; interval endpoints are the 5th and 95th percentiles.


,estimated_standard_deviation,90%_bootstrap_CI_lower,90%_bootstrap_CI_upper
feature,,,
min_avg_rss12,9.5700,8.2642,10.7953
max_avg_rss12,4.3944,3.3962,5.3200
mean_avg_rss12,5.3357,4.6965,5.8866
median_avg_rss12,5.4401,4.7963,5.9842
std_avg_rss12,1.7722,1.5726,1.9468
q1_avg_rss12,6.1536,5.5966,6.6329
q3_avg_rss12,5.1389,4.3507,5.8416
min_var_rss12,0.0000,0.0000,0.0000
max_var_rss12,5.0627,4.6090,5.4046


#### iv. Select three important feature types

I select minimum, mean, and maximum for each sensor series. Together they describe the observed lower level, typical level, and upper level of activity. This is also one of the feature sets suggested in the assignment. The selection is interpretable and preserves the amplitude profile across all six sensors.

## 2. ISLR 3.7.4

### (a) True relationship is linear: compare training RSS

The cubic regression has training RSS no greater than the linear regression. A linear function is a special case of a cubic polynomial (the quadratic and cubic coefficients can be zero), so the cubic model's larger model space contains every linear fit and can match or improve its training fit.

### (b) True relationship is linear: compare test RSS

The linear regression is expected to have lower test RSS. It matches the true functional form and estimates fewer parameters, while the cubic model can fit random training noise and increase test error. Particular finite samples can differ, but the linear model is the expected choice under the stated assumption.

### (c) True relationship is not linear: compare training RSS

The cubic regression has training RSS no greater than the linear regression because it includes the linear model as a special case. Its additional flexibility can reduce training RSS by fitting curvature in the observed sample.

### (d) True relationship is not linear: compare test RSS

There is not enough information to know which model has lower test RSS. The answer depends on how far the true relationship is from linear and whether a cubic polynomial approximates it well. The cubic model may predict better for substantial cubic-like curvature; the linear model may do better when the departure from linearity is slight or the cubic fit adds too much variance.

### References

- Li et al. (2020), [An evaluation of time series summary statistics as features for clinical prediction tasks](https://bmcmedinformdecismak.biomedcentral.com/articles/10.1186/s12911-020-1063-x).
- [ISLR 3.7.4](https://botlnec.github.io/islp/sols/chapter3/exercise4/) - linear versus cubic regression and training/test RSS.
